# Regressão em dados de NIRS (Near-Infrared Spectroscopy)

## Lidando com dados de alta dimensionalidade

Faremos a predição da quantidade de amido no milho através de dados de espectroscopia de infravermelho próximo.

Utilizaremos um conjunto de dados adaptado de [eigenvector](http://www.eigenvector.com/data/Corn/). O conjunto de dados conta com 80 instâncias de milho que foram medidas com 3 sensores NIRS entre as frequências de 1100-2498 nm, em intervalos de 2 nm. Assim, o conjunto final conta com muitos atributos preditores, 80 instâncias e 1 atributo alvo, que no nosso caso, é a quantidade de amido ("Starch").

Nesta prática veremos como até mesmo algoritmos sofisticados de AM podem ter um desempenho insatisfatório caso um bom pré-processamento não seja aplicado.

Iniciamos importando algumas bibliotecas úteis!

In [ ]:
import numpy as np # Manipulação matricial, operações algébricas e estatísticas vetorizadas
import pandas as pd # Leitura de dados
from sklearn.ensemble import RandomForestRegressor # Random Forest
from sklearn.preprocessing import StandardScaler # Padronização de dados

np.random.seed(0) # Seed de geração de números aleatórios para garantir reproducibilidade

### Leitura dos dados

In [ ]:
data = pd.read_csv("corn.csv", index_col="sample_id")
data

### Embaralhamento dos dados
Uma boa prática para dados que não tem dependência temporal (ou não deveria ter).

In [ ]:
# Shuffle dataset
data = data.sample(frac=1, random_state=543)  # TODO: descobrir pq nao funciona o seed fixo

print('Dimensões:', data.shape)
data.iloc[:2]

## Separação em porções de treino e teste, treinamento de modelo e avaliação

Aqui faremos uma análise de desempenho muito simples. A ideia é separar o conjunto em porções de treino e teste. O teste nunca será utilizado para treinar o modelo, servindo apenas como indicativo de desempenho. Outras estratégia são preferíveis em situações reais para avaliação de desempenho, como por exemplo, a validação cruzada. No entanto, esse não é o nosso foco aqui. Passos a serem desenvolvidos:

- Separação do conjunto em porções de treino e teste (80/20)
- Definição de medidas de avaliação
- Treinamento de regressor
- Avaliação

Primeiramente, tomaremos 80% das instâncias para treinamento do modelo.

In [ ]:
threshold = int(round(0.8 * len(data)))
print('Utilizaremos {} instâncias para treino!'.format(threshold))

### Partições de treino e teste

In [ ]:
y = data["Starch"]
X = data.drop(columns=["Starch"])
X.head()

X_tr = X.iloc[:threshold, :].values
y_tr = y.iloc[:threshold].values

X_ts = X.iloc[threshold:, :].values
y_ts = y.iloc[threshold:].values

print(X_tr.shape)
print(y_tr.shape)

print(X_ts.shape)
print(y_ts.shape)


Separação das features de treinamento e o atributo alvo.

### Antes de continuar...

Como avaliaremos nosso futuro regressor? No conjunto de teste, certo? Com que medidas de avaliação? Será que algumas são mais adequadas do que as outras dependendo do contexto?

Avaliaremos duas possibilidades:
- Root Mean Square Error (RMSE)
- Relative Root Mean Square Error (RRMSE)

#### RMSE

Mede a raiz do desvio quadrático da previsão ($\hat{y}$) para os valores esperados ($y$). Os valores resultantes estão na mesma escala do que o atributo alvo.

\begin{equation*}
    \text{RMSE} = \sqrt{\frac{\sum_{i}^{n}(y_i - \hat{y}_i)^2}{n}}
\end{equation*}

#### RRMSE 

Compara o RMSE do preditor contra um baseline (que é a média da variável alvo). Não possui escala, mas permite interpretações interessantes. Por exemplo, um $\text{RRMSE} \ge 1$ implica que o regressor é pior ou tão ruim quanto o baseline (sempre "chutar" na média).

\begin{equation*}
    \text{RRMSE} = \sqrt{\frac{\sum_{i}^{n}(y_i - \hat{y}_i)^2}{\sum_{i}^{n} (y_i - \overline{y})^2}}
\end{equation*}

Implementações (normalmente se usa uma biblioteca com as funções prontas, para suas análises a entregar na tarefa use a medida R2):

In [ ]:
def RMSE(obs, pred):
    return np.sqrt(np.sum((obs - pred) ** 2) / len(obs))

def RRMSE(obs, pred):
    # Adicionamos uma pequena constante (10^{-6}) no numerador e denominador
    # para evitar possíveis divisões por zero
    num = np.sum((obs - pred) ** 2) + 1e-6
    den = np.sum((obs - np.mean(obs)) ** 2) + 1e-6
    
    return np.sqrt(num / den)

### Voltando ao regressor

Alguns fatos interessantes sobre a Random Forest (RF) e porque está é a minha técnica de escolha em grande parte dos casos:

- Ensemble
- Robusta a ruídos e outliers
- Pouco sensível a parametrização (na prática, sempre "ready to go")
- Inerentemente paralelizável
- Uma das técnicas mais acuradas atualmente

Antes de treinar a RF aplicaremos uma padronização aos dados (z-score). "Centering" pela média e "scaling" pelo desvio padrão $\rightarrow$ dados com média zero e desvio padrão unitário. Não é mandatório para todos os métodos, mas geralmente recomendado. Para algoritmos de árvores de decisão convencionais e respectivos comitês, não faz diferença. Pondere a respeito.

\begin{equation*}
    z_i = \frac{x_i - \overline{x}}{\sigma_x}
\end{equation*}


### Padronizando dados e treinando uma RF:

In [ ]:
scaler_x = StandardScaler()
scaler_y = StandardScaler()
X_tr = scaler_x.fit_transform(X_tr)
y_tr = scaler_y.fit_transform(y_tr.reshape(-1, 1))[:, 0]

rf = RandomForestRegressor(n_estimators=10, max_features='sqrt')
rf.fit(X_tr, y_tr)

### Avaliando o modelo induzido

Aqui, **aplicamos** a mesma estratégia de padronização nos dados de teste antes de submetê-los ao modelo treinado.

In [ ]:
X_ts = scaler_x.transform(X_ts)
predictions_orig = rf.predict(X_ts)
predictions_orig = scaler_y.inverse_transform(predictions_orig.reshape(-1, 1))[:, 0]

In [ ]:
predictions_orig

E calculamos os erros obtidos:


#### RMSE

In [ ]:
print('RMSE:', RMSE(y_ts, predictions_orig))

Será que estamos indo bem? Que tal olhar os ranges?

In [ ]:
print('Range dos dados de treinamento:', max(y_tr) - min(y_tr))
print('Range dos dados de teste:', max(y_ts) - min(y_ts))

Parece que o erro aparentemente baixo não foi tão baixo assim.

Agora avaliaremos o nosso modelo na presença de um baseline:

#### RRMSE

In [ ]:
print('RRMSE:', RRMSE(y_ts, predictions_orig))

### Conclusão:

Teria sido melhor sempre chutar a média do conjunto de teste. :/

Mas e agora? O que fazer? Por que um modelo dito robusto e acurado se saiu tão mal?

O que estamos deixando passar?

#### Hipóteses:

- Dados? Muito pequena a amostra para treinamento? $\rightarrow$ pode ser um caminho...
- Forma como tratamos os dados $\rightarrow$ pré-processamento?
    * Muito provável!


Vamos voltar ao princípio, analisando os dados:

In [ ]:
# Dimensões dos dados de treinamento
X_tr.shape

### Algo estranho?

Sim!!! Temos muito mais descritores do que instâncias. Uma alternativa pode ser utilizar um extrator de features, já que nesse contexto estamos mais interessados em predizer a propriedade do que interpretar os atributos que estamos utilizando para tal tarefa.

Utilizaremos a Principal Component Analysis ([PCA](https://towardsdatascience.com/a-step-by-step-explanation-of-principal-component-analysis-b836fb9c97e2)) para esse fim!


In [ ]:
import matplotlib.pyplot as plt # Biblioteca para a construção de gráficos
from sklearn.decomposition import PCA

In [ ]:
# Definindo novos conjuntos de treinamento
X_tr_pca = X_tr[:, :-1]
X_ts_pca = X_ts[:, :-1]

scaler_x_pca = StandardScaler()
X_tr_pca = scaler_x_pca.fit_transform(X_tr_pca)
X_ts_pca = scaler_x_pca.transform(X_ts_pca)

### Vamos fazer alguns testes com a PCA

In [ ]:
# Começamos chutando alto => n_components == número de instâncias (máximo permitido pelo sklearn)
pca = PCA()
pca.fit(X_tr_pca)

Antes de utilizarmos tantos componentes, que tal avaliarmos a **variância dos dados** que cada um deles explica (em %)?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot([i + 1 for i in range(len(pca.explained_variance_ratio_))], 100 * pca.explained_variance_ratio_, 'b-')
ax.set_xlabel('Componentes')
ax.set_ylabel('% da variância explicada')
pass

#### Parece haver uma queda vertiginosa! Vamos explorar do ponto de vista cumulativo.

In [ ]:
print('Primeiros elementos', (100 * pca.explained_variance_ratio_.cumsum())[:20])
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot([i + 1 for i in range(len(pca.explained_variance_ratio_))], 100 * pca.explained_variance_ratio_.cumsum(), 'b-')
ax.axvline(2, color='green')
ax.axvline(10, color='yellow')
ax.axvline(20, color='red')
ax.set_xlabel('Componentes')
ax.set_ylabel('% da variância explicada')
pass

### Vamos tentar utilizar menos componentes?

Normalmente, buscamos a quantidade de componentes que explicam entre 95-99% da variância dos dados (experiência própria). O restante normalmente está capturando variações de ruído ou outros aspectos que não nos interessam. Nesse caso empíricamente escolhi os 10 primeiros componentes (um pouco exagerado até, talvez). Os resultados foram bem melhores para a RF. Decidi adicionar um modelo linear e aumentar para 15 e depois 20 componentes. Os resultados surpreenderam. 

Sugiro testarem outros valores e avaliarem os resultados obtidos.


In [ ]:
pca = PCA(n_components=20)

X_tr_pca = pca.fit_transform(X_tr_pca)
X_ts_pca = pca.transform(X_ts_pca)

In [ ]:
X_tr_pca.shape

#### Voltando à RF

In [ ]:
rf_pca = RandomForestRegressor(n_estimators=100, max_features='sqrt')
rf_pca.fit(X_tr_pca, y_tr)
preds_rf = rf_pca.predict(X_ts_pca)
preds_rf = scaler_y.inverse_transform(preds_rf.reshape(-1, 1))[:, 0]

print('RMSE da RF:', RMSE(y_ts, preds_rf))
print('RRMSE da RF:', RRMSE(y_ts, preds_rf))

#### Momento da simplicidade

Vamos treinar uma simples regressão linear

In [ ]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_tr_pca, y_tr)

preds_lr = lr.predict(X_ts_pca)
preds_lr = scaler_y.inverse_transform(preds_lr.reshape(-1, 1))[:, 0]
print('RMSE da LR:', RMSE(y_ts, preds_lr))
print('RRMSE da LR:', RRMSE(y_ts, preds_lr))

In [ ]:
lr.coef_

# Considerações finais

Um modelo linear se saiu melhor do que um ensemble!
- Canhão vs. mosca.
- Pequena quantidade de instâncias de treinamento.
- Problema linear! $\rightarrow$ viés de aprendizado do algoritmo não adequado ao problema, dadas as características do dataset.
    * Como traçar uma reta usando retângulos? :D

A utilização de pré-processamento com PCA foi essencial para diminuir o erro preditivo!
- Os modelos finais conseguiram melhor e muito o desempenho do baseline
    * Importância da adoção de medidas adequadas para avaliação.